In [ ]:
import json
import glob
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

# Annotation floor for IEEE-style figures. The previous revision hardcoded
# fontsize=8 inside the label helper -- identical to the base font size --
# which is why value labels rendered as large as axis labels and collided
# with the marks they annotated. Every figure routes its labels through
# ANNOT_SIZE now, so the size lives in exactly one place.
ANNOT_SIZE = 7
NOTE_SIZE = 6.5

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 8,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "figure.dpi": 300,
    "savefig.dpi": 300,
})

# IEEEtran geometry: \columnwidth = 3.5in, \textwidth = 7.16in. Figures are
# authored at final size so LaTeX never rescales them (rescaling is what
# makes font sizes inconsistent across figures in a paper).
COL_W = 3.5
TEXT_W = 7.16

OUTDIR = Path.cwd()

# --- Okabe-Ito colourblind-safe palette -------------------------------
COLOR_FULL = "#0072B2"       # blue
COLOR_STRIPPED = "#D55E00"   # vermillion
COLOR_CONNECT = "#B8B8B8"    # dumbbell connector
COLOR_ROWGUIDE = "#EDEDED"
COLOR_NA = "#909090"

GROUP_COLORS = {
    "Baselines": "#009E73",           # bluish green
    "Fine-tuned LLM": "#E69F00",      # orange -- makes Qwen pop as the outlier
    "Frontier zero-shot": "#CC79A7",  # reddish purple
}

# --- Canonical model grouping. Qwen is NOT a baseline: it is the model H1
# fine-tunes, so it gets its own group. Fixed order, used identically in
# every figure. ------------------------------------------------------
GROUPS = [
    ("Baselines", [
        "TF-IDF (1-2gram) + LinearSVC",
        "google-bert/bert-base-uncased",
        "mental/mental-bert-base-uncased",
    ]),
    ("Fine-tuned LLM", [
        "Qwen/Qwen2.5-3B-Instruct",
    ]),
    ("Frontier zero-shot", [
        "anthropic/claude-sonnet-5",
        "openai/gpt-5.6-luna",
    ]),
]
CANONICAL_MODELS = [m for _, models in GROUPS for m in models]

DISPLAY_NAMES = {
    "TF-IDF (1-2gram) + LinearSVC": "SVM (TF-IDF)",
    "google-bert/bert-base-uncased": "BERT-base",
    "mental/mental-bert-base-uncased": "MentalBERT",
    "Qwen/Qwen2.5-3B-Instruct": "Qwen2.5-3B",
    "anthropic/claude-sonnet-5": "Claude Sonnet 5",
    "openai/gpt-5.6-luna": "GPT-5.6 Luna",
}

CLASS_DISPLAY_NAMES = {
    "depression": "Depression",
    "SuicideWatch": "Suicidal Ideation",
    "Anxiety": "Anxiety",
    "bipolar": "Bipolar Disorder",
}
CLASSES = list(CLASS_DISPLAY_NAMES.keys())

# unparseable_rate is only meaningful for models that generate free text --
# a classification head always emits a valid class.
NON_GENERATIVE = {"SVM (TF-IDF)", "BERT-base", "MentalBERT"}

NA_NOTE = "no stripped run"


def retention_ratio(full, stripped):
    """Fraction of full-text performance surviving keyword stripping.

    Reported alongside the absolute drop because it normalises for baseline
    strength -- it answers the reviewer objection that a model only drops
    further because it started higher."""
    if full is None or stripped is None or full == 0:
        return None
    return stripped / full

In [ ]:
def normalize_model_name(raw):
    """bert-base-uncased's stripped-variant JSON drops the 'google-bert/'
    org prefix that its full-variant JSON has -- fold both to one key so
    GROUPS/DISPLAY_NAMES only need to list the model once."""
    if raw == "bert-base-uncased":
        return "google-bert/bert-base-uncased"
    return raw


def load_results(results_dir="."):
    results = {}
    for path in sorted(glob.glob(os.path.join(results_dir, "results_*.json"))):
        with open(path) as f:
            d = json.load(f)
        canonical = normalize_model_name(d["model"])
        results[(canonical, d["variant"])] = {**d, "_source_path": path, "_canonical_model": canonical}
    return results


results = load_results(".")
print("Loaded:", [f"{DISPLAY_NAMES.get(m, m)}_{v}" for (m, v) in results.keys()])

In [ ]:
# ---------------------------------------------------------------------
# Shared drawing helpers for the horizontal dumbbell / dot-plot figures.
#
# Why dumbbells rather than grouped bars: a bar encodes magnitude as
# LENGTH, so truncating its axis at 0.5 (necessary here, or every bar is a
# near-identical tall block) genuinely distorts the comparison and forces a
# "not to scale from zero" caption disclaimer. A dot encodes value as
# POSITION, which carries no zero-baseline promise, so the truncated range
# is legitimate. The connector between the pair of dots has length equal to
# the full-vs-stripped drop -- the paper's actual claim becomes the primary
# visual, instead of something the reader must compute by comparing two bar
# heights.
# ---------------------------------------------------------------------

def build_rows(models):
    """Interleave group headers with model rows so the grouping is carried
    by the y-axis itself, rather than by dividers drawn under the plot."""
    rows = []
    for gname, gmodels in GROUPS:
        present = [m for m in gmodels if m in models]
        if not present:
            continue
        rows.append(("header", gname))
        rows.extend(("model", m) for m in present)
    return rows


def setup_rows(ax, rows, xlim, row_guides=True):
    """Lay out header/model rows top-to-bottom and return {model: y}."""
    ypos = {}
    labels = []
    for i, (kind, key) in enumerate(rows):
        labels.append(DISPLAY_NAMES[key] if kind == "model" else key)
        if kind == "model":
            ypos[key] = i

    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(labels)
    # Group headers are right-aligned like every other tick label, so their
    # extra length makes them overhang to the left -- which reads as the
    # hierarchy it is, with no indentation hacks.
    for tick, (kind, _) in zip(ax.get_yticklabels(), rows):
        if kind == "header":
            tick.set_style("italic")
            tick.set_color("#444444")
            tick.set_fontsize(7.5)

    ax.set_ylim(len(rows) - 0.5, -0.5)   # inverted: first row at the top
    ax.tick_params(axis="y", length=0)
    ax.set_xlim(*xlim)
    ax.xaxis.grid(True, color="#CCCCCC", linewidth=0.4, linestyle=":", alpha=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)

    if row_guides:
        for y in ypos.values():
            ax.plot(xlim, [y, y], color=COLOR_ROWGUIDE, linewidth=0.6,
                    zorder=0, solid_capstyle="butt")
    return ypos


def needs_stacked_labels(low, high, xlim):
    """Decide whether an outboard label pair would collide.

    Two ways it can go wrong:
      1. The dots are nearly coincident, so the left and right labels run
         into each other (GPT-5.6 Luna unparseable: 15.1% vs 14.9%).
      2. The lower dot sits near the left edge, so its left-hand label
         would overrun the axis into the tick labels (Qwen2.5-3B
         unparseable: 0.6%, almost on the axis floor).
    Either case falls back to stacking both labels to the right of the
    higher dot, where there is always room."""
    span = xlim[1] - xlim[0]
    return (high - low) < 0.09 * span or (low - xlim[0]) < 0.12 * span


def annotate_pair(ax, y, low, high, xlim, fmt, colors):
    """Label a dot pair outboard: lower value to the left, higher to the
    right, so a label can never sit on top of the marks."""
    span = xlim[1] - xlim[0]
    pad = 0.012 * span
    c_low, c_high = colors

    if needs_stacked_labels(low, high, xlim):
        ax.text(high + pad, y - 0.24, fmt(high), ha="left", va="center",
                fontsize=ANNOT_SIZE, color=c_high)
        ax.text(high + pad, y + 0.24, fmt(low), ha="left", va="center",
                fontsize=ANNOT_SIZE, color=c_low)
    else:
        ax.text(low - pad, y, fmt(low), ha="right", va="center",
                fontsize=ANNOT_SIZE, color=c_low)
        ax.text(high + pad, y, fmt(high), ha="left", va="center",
                fontsize=ANNOT_SIZE, color=c_high)


def draw_dumbbell_row(ax, y, full, stripped, xlim, fmt, ms=4.2):
    """One model row: full dot, stripped dot, connector, labels.

    Claude Sonnet 5 has no stripped run, so it gets a lone dot plus an
    explicit note. The note goes on whichever side of the dot has room,
    which for a high full-text score (0.846) is the left."""
    span = xlim[1] - xlim[0]
    pad = 0.012 * span

    if stripped is None:
        ax.plot([full], [y], "o", color=COLOR_FULL, markersize=ms, zorder=3)
        mid = xlim[0] + span / 2
        if full > mid:
            ax.text(full - pad, y, fmt(full), ha="right", va="center",
                    fontsize=ANNOT_SIZE, color=COLOR_FULL)
            ax.text(full - pad - 0.10 * span, y, NA_NOTE, ha="right",
                    va="center", fontsize=NOTE_SIZE, style="italic", color=COLOR_NA)
        else:
            ax.text(full + pad, y, fmt(full), ha="left", va="center",
                    fontsize=ANNOT_SIZE, color=COLOR_FULL)
            ax.text(full + pad + 0.10 * span, y, NA_NOTE, ha="left",
                    va="center", fontsize=NOTE_SIZE, style="italic", color=COLOR_NA)
        return

    ax.plot([stripped, full], [y, y], color=COLOR_CONNECT, linewidth=2.2,
            solid_capstyle="round", zorder=2)
    ax.plot([stripped], [y], "o", color=COLOR_STRIPPED, markersize=ms, zorder=3)
    ax.plot([full], [y], "o", color=COLOR_FULL, markersize=ms, zorder=3)

    low, high = (stripped, full) if stripped <= full else (full, stripped)
    c_low = COLOR_STRIPPED if stripped <= full else COLOR_FULL
    c_high = COLOR_FULL if stripped <= full else COLOR_STRIPPED
    annotate_pair(ax, y, low, high, xlim, fmt, (c_low, c_high))


DUMBBELL_LEGEND = [
    Line2D([], [], marker="o", linestyle="none", color=COLOR_FULL,
           markersize=4.2, label="Full text"),
    Line2D([], [], marker="o", linestyle="none", color=COLOR_STRIPPED,
           markersize=4.2, label="Keyword-stripped"),
    Line2D([], [], color=COLOR_CONNECT, linewidth=2.2, label="Drop"),
]


def save_fig_verified(fig, stem, records):
    """Print every plotted value next to its source JSON path, then export
    BOTH a vector PDF (for LaTeX) and a raster PNG (for sharing) from the
    same call -- previously the PNGs were written by a separate run and
    silently drifted out of date relative to the PDFs.

    bbox_inches='tight' is deliberately NOT used: it crops the saved canvas
    to content extents, shrinking the PDF page below the figsize we set and
    reintroducing the arbitrary-size-scaled-later problem. Figures use
    layout='constrained' instead, which fits content INTO the fixed canvas."""
    print(f"--- {stem} ---")
    for r in records:
        print(f"  {r.get('label',''):28s} model={r.get('model',''):32s} "
              f"value={r.get('value')}  <- {r.get('source','')}")
    for ext in ("pdf", "png"):
        fig.savefig(OUTDIR / f"{stem}.{ext}", format=ext)
    plt.show()
    plt.close(fig)

In [ ]:
# =====================================================================
# Figure 1 -- f1_comparison: two-panel dumbbell.
# Merges the previously separate anxiety_f1_comparison and
# macro_f1_comparison figures, which had identical structure and repeated
# the same model list, group dividers and legend twice.
# =====================================================================

XLIM_F1 = (0.42, 0.95)
fmt3 = lambda v: f"{v:.3f}"

fig, axes = plt.subplots(1, 2, figsize=(TEXT_W, 3.1), sharey=True,
                         layout="constrained")
rows = build_rows(CANONICAL_MODELS)
records = []

panels = [("anxiety_f1", "Anxiety-class F1", "(a)"),
          ("macro_f1", "Macro F1", "(b)")]

for ax, (metric_key, xlabel, panel) in zip(axes, panels):
    ypos = setup_rows(ax, rows, XLIM_F1)
    ax.set_xticks([0.5, 0.6, 0.7, 0.8, 0.9])

    for m in CANONICAL_MODELS:
        full_rec = results.get((m, "full"))
        stripped_rec = results.get((m, "stripped"))
        full = full_rec[metric_key] if full_rec else None
        stripped = stripped_rec[metric_key] if stripped_rec else None
        records.append({"label": f"{metric_key} full", "model": m,
                        "value": full if full is not None else "n/a",
                        "source": full_rec["_source_path"] if full_rec else ""})
        records.append({"label": f"{metric_key} stripped", "model": m,
                        "value": stripped if stripped is not None else "n/a",
                        "source": stripped_rec["_source_path"] if stripped_rec else ""})
        if full is None:
            continue
        draw_dumbbell_row(ax, ypos[m], full, stripped, XLIM_F1, fmt3)

    ax.set_xlabel(xlabel)
    ax.set_title(panel, loc="left", fontsize=9, fontweight="bold", pad=3)

fig.legend(handles=DUMBBELL_LEGEND, loc="outside lower center", ncol=3,
           frameon=False, handletextpad=0.4, columnspacing=1.6)

save_fig_verified(fig, "f1_comparison", records)

In [ ]:
# =====================================================================
# Figure 2 -- lexical_dependence: horizontal lollipop, sorted by drop.
# Horizontal orientation removes the 30-degree rotated tick labels; the
# stems start at a true zero, so no truncation caveat is needed here.
# Each row carries the absolute drop AND the retention ratio.
# =====================================================================

drops = []
for m in CANONICAL_MODELS:
    full_rec = results.get((m, "full"))
    stripped_rec = results.get((m, "stripped"))
    if full_rec is None or stripped_rec is None:
        continue  # Claude Sonnet 5: no stripped run, no delta to plot
    group = next(g for g, models in GROUPS if m in models)
    drops.append({
        "model": m,
        "delta": full_rec["anxiety_f1"] - stripped_rec["anxiety_f1"],
        "retained": retention_ratio(full_rec["anxiety_f1"], stripped_rec["anxiety_f1"]),
        "group": group,
        "sources": f"{full_rec['_source_path']} & {stripped_rec['_source_path']}",
    })

drops.sort(key=lambda d: d["delta"], reverse=True)

XLIM_DROP = (0, 0.40)
fig, ax = plt.subplots(figsize=(COL_W, 2.1), layout="constrained")

for i, d in enumerate(drops):
    c = GROUP_COLORS[d["group"]]
    ax.plot(XLIM_DROP, [i, i], color=COLOR_ROWGUIDE, linewidth=0.6, zorder=0)
    ax.plot([0, d["delta"]], [i, i], color=c, linewidth=1.6,
            solid_capstyle="butt", zorder=2)
    ax.plot([d["delta"]], [i], "o", color=c, markersize=4.2, zorder=3)
    ax.text(d["delta"] + 0.012, i, f"{d['delta']:.3f}", ha="left", va="center",
            fontsize=ANNOT_SIZE, color="#222222")
    # Retention sits in a fixed right-hand column, greyed so the eye reads
    # it as a secondary quantity rather than a second bar label.
    ax.text(1.0, i, f"{d['retained']:.0%} retained", ha="right", va="center",
            fontsize=NOTE_SIZE, color="#666666",
            transform=ax.get_yaxis_transform())

ax.set_yticks(range(len(drops)))
ax.set_yticklabels([DISPLAY_NAMES[d["model"]] for d in drops])
ax.set_ylim(len(drops) - 0.5, -0.5)
ax.tick_params(axis="y", length=0)
ax.set_xlim(*XLIM_DROP)
ax.set_xticks([0.0, 0.1, 0.2, 0.3])
ax.set_xlabel("Anxiety-class F1 drop  (full − stripped)")
ax.xaxis.grid(True, color="#CCCCCC", linewidth=0.4, linestyle=":", alpha=0.8)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)

# Legend goes outside the axes: rows are re-sorted by drop here, so group
# membership is not readable from position the way it is in Figure 1, but
# any in-axes placement lands on the retention column.
fig.legend(handles=[Line2D([], [], color=c, linewidth=1.6, marker="o",
                           markersize=4.2, label=g)
                    for g, c in GROUP_COLORS.items()],
           loc="outside lower center", ncol=3, frameon=False,
           fontsize=NOTE_SIZE, handletextpad=0.4, columnspacing=1.2)

save_fig_verified(fig, "lexical_dependence",
                  [{"label": "anxiety_f1 delta", "model": d["model"],
                    "value": f"{d['delta']:.4f} ({d['retained']:.1%} retained)",
                    "source": d["sources"]} for d in drops])

In [ ]:
# =====================================================================
# Figure 3 -- per_class_f1_heatmap: three panels.
# Panels (a)/(b) share one sequential scale; panel (c) is new and shows
# the per-class delta on a diverging scale centred at zero, so the reader
# can see WHICH classes collapse under stripping instead of subtracting
# two panels by eye.
# =====================================================================

fig, (ax_full, ax_strip, ax_delta) = plt.subplots(
    1, 3, figsize=(TEXT_W, 2.9), sharey=True, layout="constrained")

cmap = plt.get_cmap("cividis").copy()
cmap.set_bad(COLOR_ROWGUIDE)
dmap = plt.get_cmap("RdBu_r").copy()
dmap.set_bad(COLOR_ROWGUIDE)

records = []
mats = {}
for variant in ("full", "stripped"):
    mat = np.full((len(CANONICAL_MODELS), len(CLASSES)), np.nan)
    for i, m in enumerate(CANONICAL_MODELS):
        rec = results.get((m, variant))
        for j, c in enumerate(CLASSES):
            if rec is not None:
                mat[i, j] = rec["full_report"][c]["f1-score"]
                records.append({"label": f"{c} {variant}", "model": m,
                                "value": mat[i, j], "source": rec["_source_path"]})
            else:
                records.append({"label": f"{c} {variant}", "model": m,
                                "value": "n/a", "source": ""})
    mats[variant] = mat
mats["delta"] = mats["full"] - mats["stripped"]

DMAX = 0.26  # symmetric limit; largest observed drop is Qwen anxiety 0.254

# Every per-class F1 in this study falls in [0.40, 0.85]. Ramping the
# sequential scale over the full [0, 1] therefore compresses all the real
# variation into the middle third of cividis and the panels read as
# near-uniform olive. Clamping to the occupied range restores the
# discrimination the panel exists to show; no value is clipped, and the
# printed number in each cell carries the exact magnitude regardless.
FMIN, FMAX = 0.35, 0.90


def annotate_cells(ax, mat, is_delta):
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if np.isnan(v):
                ax.text(j, i, "n/a", ha="center", va="center",
                        fontsize=NOTE_SIZE, style="italic", color="#555555")
                continue
            if is_delta:
                colour = "white" if abs(v) > 0.62 * DMAX else "black"
                # Avoid rendering a rounded-to-zero value as "-0.00".
                label = "0.00" if abs(v) < 0.005 else f"{v:+.2f}"
            else:
                colour = "white" if v < FMIN + 0.45 * (FMAX - FMIN) else "black"
                label = f"{v:.2f}"
            ax.text(j, i, label, ha="center", va="center", fontsize=6.5, color=colour)


panels = [
    (ax_full, "full", "Full text", "(a)", False),
    (ax_strip, "stripped", "Keyword-stripped", "(b)", False),
    (ax_delta, "delta", "Drop (full − stripped)", "(c)", True),
]

for ax, key, xlabel, panel, is_delta in panels:
    masked = np.ma.masked_invalid(mats[key])
    if is_delta:
        im_d = ax.imshow(masked, cmap=dmap, vmin=-DMAX, vmax=DMAX, aspect="auto")
    else:
        im_s = ax.imshow(masked, cmap=cmap, vmin=FMIN, vmax=FMAX, aspect="auto")
    ax.tick_params(axis="both", length=0)
    ax.set_xticks(range(len(CLASSES)))
    ax.set_xticklabels([CLASS_DISPLAY_NAMES[c] for c in CLASSES],
                       rotation=30, ha="right", fontsize=7)
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_title(panel, loc="left", fontsize=9, fontweight="bold", pad=3)
    annotate_cells(ax, mats[key], is_delta)

    # Group boundaries: models are rows here, so the row-band treatment
    # from the dumbbell figures becomes horizontal rules between groups.
    pos = 0
    for gi, (gname, models) in enumerate(GROUPS):
        if gi > 0:
            ax.axhline(pos - 0.5, color="white", linewidth=1.4, zorder=4)
        pos += len(models)

# Model names appear on panel (a) only -- the previous two-panel version
# repeated them and paid wspace=0.7 to fit, which is the width panel (c)
# now occupies.
ax_full.set_yticks(range(len(CANONICAL_MODELS)))
ax_full.set_yticklabels([DISPLAY_NAMES[m] for m in CANONICAL_MODELS], fontsize=8)
ax_full.tick_params(axis="y", length=0)

cb1 = fig.colorbar(im_s, ax=[ax_full, ax_strip], location="bottom",
                   fraction=0.06, pad=0.04, aspect=45)
cb1.set_ticks([0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
cb1.set_label("F1 score", fontsize=8)
cb1.ax.tick_params(labelsize=7)
cb2 = fig.colorbar(im_d, ax=[ax_delta], location="bottom",
                   fraction=0.06, pad=0.04, aspect=22)
cb2.set_label("F1 drop", fontsize=8)
cb2.ax.tick_params(labelsize=7)

save_fig_verified(fig, "per_class_f1_heatmap", records)

In [ ]:
# =====================================================================
# Figure 4 -- robustness_rate: unparseable output rate.
# Same dumbbell grammar as Figure 1 so the reader learns one encoding.
# Only five real numbers here, a density at which bars overstate.
# =====================================================================

generative_models = [m for m in CANONICAL_MODELS if DISPLAY_NAMES[m] not in NON_GENERATIVE]

XLIM_UNP = (-0.012, 0.21)
fmt_pct = lambda v: f"{v:.1%}"

fig, ax = plt.subplots(figsize=(COL_W, 1.7), layout="constrained")
rows = build_rows(generative_models)
ypos = setup_rows(ax, rows, XLIM_UNP)
records = []

for m in generative_models:
    full_rec = results.get((m, "full"))
    stripped_rec = results.get((m, "stripped"))
    full = full_rec["unparseable_rate"] if full_rec else None
    stripped = stripped_rec["unparseable_rate"] if stripped_rec else None
    records.append({"label": "unparseable_rate full", "model": m,
                    "value": full if full is not None else "n/a",
                    "source": full_rec["_source_path"] if full_rec else ""})
    records.append({"label": "unparseable_rate stripped", "model": m,
                    "value": stripped if stripped is not None else "n/a",
                    "source": stripped_rec["_source_path"] if stripped_rec else ""})
    if full is None:
        continue
    draw_dumbbell_row(ax, ypos[m], full, stripped, XLIM_UNP, fmt_pct)

ax.set_xticks([0.0, 0.05, 0.10, 0.15, 0.20])
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel("Unparseable output rate")

fig.legend(handles=DUMBBELL_LEGEND[:2], loc="outside lower center", ncol=2,
           frameon=False, handletextpad=0.4, columnspacing=1.6)

save_fig_verified(fig, "robustness_rate", records)

print("CAPTION NOTE (robustness_rate): SVM and encoder-based models "
      "(BERT-base, MentalBERT) are excluded -- a classification head "
      "always emits a valid class, so they are structurally immune to "
      "'unparseable' output, unlike free-text-generating LLMs.")

In [ ]:
# =====================================================================
# Self-check: assert the numbers the figures draw match the source JSONs.
# Runs on every notebook execution; fails loudly if a helper or a results
# file changes underneath the figures.
# =====================================================================

def _f1(model, variant, key="anxiety_f1"):
    return results[(model, variant)][key]


# retention_ratio contract
assert retention_ratio(0.8, 0.4) == 0.5
assert retention_ratio(0.8, None) is None
assert retention_ratio(0.0, 0.4) is None

# Claude Sonnet 5 is the only model without a stripped run.
missing = [m for m in CANONICAL_MODELS if (m, "stripped") not in results]
assert missing == ["anthropic/claude-sonnet-5"], missing
assert abs(_f1("anthropic/claude-sonnet-5", "full") - 0.8460) < 5e-4

# Drop ranking drives Figure 2's row order and the paper's central claim.
ranking = [(DISPLAY_NAMES[d["model"]], round(d["delta"], 3)) for d in drops]
assert ranking == [
    ("Qwen2.5-3B", 0.254),
    ("SVM (TF-IDF)", 0.149),
    ("BERT-base", 0.130),
    ("MentalBERT", 0.129),
    ("GPT-5.6 Luna", 0.086),
], ranking

# Retention ranks the same way, so the claim survives normalising for
# baseline strength (Qwen worst, GPT-5.6 Luna best).
retained = [(DISPLAY_NAMES[d["model"]], round(d["retained"], 3)) for d in drops]
assert retained[0] == ("Qwen2.5-3B", 0.688), retained[0]
assert retained[-1] == ("GPT-5.6 Luna", 0.893), retained[-1]
assert [r for _, r in retained] == sorted(r for _, r in retained), retained

# Every plotted F1 lies inside the dumbbell x-range, i.e. nothing is drawn
# off-canvas by the fixed limits.
for m in CANONICAL_MODELS:
    for variant in ("full", "stripped"):
        if (m, variant) not in results:
            continue
        for key in ("anxiety_f1", "macro_f1"):
            v = _f1(m, variant, key)
            assert XLIM_F1[0] < v < XLIM_F1[1], (m, variant, key, v)

# Colour limits must bracket every value they encode, or a cell would be
# clipped to the end of the ramp and silently misread.
assert np.nanmax(np.abs(mats["delta"])) <= DMAX, np.nanmax(np.abs(mats["delta"]))
for key in ("full", "stripped"):
    assert FMIN <= np.nanmin(mats[key]) and np.nanmax(mats[key]) <= FMAX, key

# Label placement: the stacking fallback must fire exactly where an
# outboard pair would collide, and nowhere else.
#   - GPT-5.6 Luna unparseable 15.1% vs 14.9%: dots nearly coincident.
#   - Qwen2.5-3B unparseable 0.6%: lower dot sits on the axis floor.
#   - every Figure 1 pair is well separated and clear of the left edge.
assert needs_stacked_labels(0.149, 0.1514, XLIM_UNP)          # coincident
assert needs_stacked_labels(0.0057, 0.0332, XLIM_UNP)         # near left edge
assert not needs_stacked_labels(0.20, 0.60, (0.0, 1.0))       # comfortably apart
for m in CANONICAL_MODELS:
    if (m, "stripped") not in results:
        continue
    for key in ("anxiety_f1", "macro_f1"):
        lo, hi = _f1(m, "stripped", key), _f1(m, "full", key)
        assert not needs_stacked_labels(lo, hi, XLIM_F1), (m, key, lo, hi)

print("All figure self-checks passed.")
print()
print("SUPERSEDED by f1_comparison.{pdf,png} -- safe to delete once the "
      "paper source is updated:")
print("  anxiety_f1_comparison.pdf / .png")
print("  macro_f1_comparison.pdf / .png")
print("  per_class_f1_bars.png  (orphan: produced by no current cell)")